# 🔗 Strongly Connected Components — Runnable Notebook

Companion to [`README.md`](README.md) and
[`15_strongly_connected_components_lesson.html`](15_strongly_connected_components_lesson.html).

Find maximal groups where every vertex reaches every other — via **Kosaraju** (2 passes) and **Tarjan** (1 pass).

## 1. The graph
`0→1→2→0` is one SCC `{0,1,2}`; `3↔4` is another `{3,4}`; the edge `2→3` links them but isn't mutual.

In [ ]:
# adj[u] = vertices u points to
adj = {0: [1], 1: [2], 2: [0, 3], 3: [4], 4: [3]}
n = 5
expected = {frozenset({0, 1, 2}), frozenset({3, 4})}
print("graph:", adj)

## 2. Kosaraju — DFS finish order, reverse edges, DFS again

In [ ]:
def kosaraju(n, adj):
    """Returns a list of SCCs (each a list of vertices)."""
    seen = [False] * n
    order = []
    def dfs1(u):
        seen[u] = True
        for v in adj[u]:
            if not seen[v]:
                dfs1(v)
        order.append(u)                       # push on FINISH (post-order)
    for i in range(n):
        if not seen[i]:
            dfs1(i)

    radj = [[] for _ in range(n)]             # build the transpose (reverse every edge)
    for u in range(n):
        for v in adj[u]:
            radj[v].append(u)

    comp = [-1] * n
    sccs = []
    def dfs2(u, bag):
        comp[u] = len(sccs)
        bag.append(u)
        for v in radj[u]:
            if comp[v] == -1:
                dfs2(v, bag)
    for u in reversed(order):                 # decreasing finish time
        if comp[u] == -1:
            bag = []
            dfs2(u, bag)
            sccs.append(bag)                  # each DFS tree on the transpose = one SCC
    return sccs

sccs_k = kosaraju(n, adj)
print("Kosaraju SCCs:", sccs_k)
assert {frozenset(s) for s in sccs_k} == expected

## 3. Tarjan — one DFS with discovery + low-link numbers

In [ ]:
import sys

def tarjan(n, adj):
    disc = [-1] * n; low = [0] * n; on_stack = [False] * n
    stack = []; sccs = []; counter = [0]
    def dfs(u):
        disc[u] = low[u] = counter[0]; counter[0] += 1
        stack.append(u); on_stack[u] = True
        for v in adj[u]:
            if disc[v] == -1:                 # tree edge: recurse, then pull up low
                dfs(v)
                low[u] = min(low[u], low[v])
            elif on_stack[v]:                 # back edge to a vertex still on the stack
                low[u] = min(low[u], disc[v])
        if low[u] == disc[u]:                 # u roots an SCC -> pop it off the stack
            comp = []
            while True:
                w = stack.pop(); on_stack[w] = False
                comp.append(w)
                if w == u:
                    break
            sccs.append(comp)
    for i in range(n):
        if disc[i] == -1:
            dfs(i)
    return sccs

sccs_t = tarjan(n, adj)
print("Tarjan SCCs  :", sccs_t)
assert {frozenset(s) for s in sccs_t} == expected
print("Kosaraju and Tarjan agree:", {frozenset(s) for s in sccs_k} == {frozenset(s) for s in sccs_t})

## 4. The condensation is a DAG
Collapse each SCC to a super-node; the result is always acyclic.

In [ ]:
def condensation_edges(n, adj, sccs):
    """Return the edges between SCCs (the condensation graph)."""
    comp_of = {}
    for idx, s in enumerate(sccs):
        for v in s:
            comp_of[v] = idx
    edges = set()
    for u in range(n):
        for v in adj[u]:
            if comp_of[u] != comp_of[v]:      # an edge crossing two different SCCs
                edges.add((comp_of[u], comp_of[v]))
    return edges

edges = condensation_edges(n, adj, sccs_k)
print("condensation edges (between SCCs):", edges)
assert len(edges) == 1                          # exactly one link: {0,1,2} -> {3,4}

## ✅ Recap
- An **SCC** = a maximal set where every vertex reaches every other (directed).
- **Kosaraju**: DFS finish order → reverse edges → DFS in that order; each tree = one SCC. `O(V+E)`, two passes.
- **Tarjan**: one DFS with **`disc` / `low`** + a stack; `low[u]==disc[u]` ⇒ SCC root. `O(V+E)`, one pass.
- **Condensation** (collapse each SCC) is always a **DAG** — the acyclic skeleton of a cyclic graph.

That closes the DSA deep dive. See [`README`](../README.md) for the whole map.